# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kratosontren/flyrank-ml-work/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [64]:
!git clone https://github.com/kratosontren/flyrank-ml-work.git
%cd flyrank-ml-work

Cloning into 'flyrank-ml-work'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 134 (delta 47), reused 93 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (134/134), 1.85 MiB | 8.61 MiB/s, done.
Resolving deltas: 100% (47/47), done.
/content/flyrank-ml-work/flyrank-ml-work/flyrank-ml-work/flyrank-ml-work


In [65]:
!pip install -q datasets huggingface_hub duckdb pyarrow

In [66]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get("HF_TOKEN")
login(token)

In [67]:
from datasets import load_dataset

clients = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_clients",
    split="train"
)

clients_df = clients.to_pandas()
clients_df.head()

,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,None,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,None,None
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,None
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,None


In [68]:
daily = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True
)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

In [69]:
first = next(iter(daily))
first.keys()

dict_keys(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'])

In [70]:
sample = pd.DataFrame(
    list(islice(daily, 5000))
)

## 1. Unit of analysis + time window

One row represents one pseudonymized content item for one pseudonymized client on one reporting date.

For development, a historical non-final slice of the warehouse was used (2025-01-27 to 2025-02-12 in the sampled data). The important requirement is to avoid using the final month (June 2026), which is reserved as a sealed outcome period.

The goal of this lane is refresh/content opportunity scoring: identifying pages that may deserve review based on historical search performance.

In [71]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
sample[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date"
    ]
].head()

,client_hash_id,content_hash_id,report_date
0,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,2025-01-27
1,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,2025-01-27
2,client_9958f0a7ae1df715,content_b4462a1b90640058,2025-01-27
3,client_9958f0a7ae1df715,content_c899aef92518c714,2025-01-27
4,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,2025-01-27


## 2. Fields: feature / label / context / excluded

### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- sessions_organic

### Label / Proxy
Pages showing signs of declining search performance and therefore potentially requiring content refresh review.

### Context
- report_date
- client_hash_id
- content_hash_id
- client_has_gsc
- client_has_ga4

### Excluded
- Future outcomes
- Final month observations (June 2026)
- Any variables directly derived from future performance

These fields are excluded because they introduce information leakage and would not be available at the decision moment.

In [72]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
sample.columns.tolist()

['report_date',
 'client_hash_id',
 'content_hash_id',
 'client_has_gsc',
 'client_has_ga4',
 'gsc_data_available',
 'ga4_data_available',
 'gsc_impressions',
 'gsc_clicks',
 'gsc_sum_position',
 'gsc_avg_position',
 'ga4_pageviews',
 'ga4_sessions',
 'ga4_users',
 'ga4_engaged_sessions',
 'ga4_total_engagement_sec',
 'sessions_organic',
 'sessions_direct',
 'sessions_referral',
 'sessions_social',
 'sessions_paid',
 'sessions_ai',
 'ai_chatgpt',
 'ai_perplexity',
 'ai_gemini',
 'ai_copilot',
 'ai_claude',
 'ai_meta',
 'ai_other',
 'scroll_events']

## 3. Verify it with queries (grain, counts, missing values, windows)

The grain appears to be one client × one content item × one reporting date.

The sampled slice contains 5,000 rows spanning 2025-01-27 to 2025-02-12.

Availability filtering confirms that only a subset of rows have Search Console data available for downstream modeling.

In [73]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
sample[
    [
        "client_hash_id",
        "content_hash_id",
        "report_date"
    ]
].head()

,client_hash_id,content_hash_id,report_date
0,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,2025-01-27
1,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,2025-01-27
2,client_9958f0a7ae1df715,content_b4462a1b90640058,2025-01-27
3,client_9958f0a7ae1df715,content_c899aef92518c714,2025-01-27
4,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,2025-01-27


In [74]:
print("Rows:", len(sample))
print("Min date:", sample["report_date"].min())
print("Max date:", sample["report_date"].max())

Rows: 5000
Min date: 2025-01-27
Max date: 2025-02-12


In [75]:
sample[
    sample["gsc_data_available"] == True
].shape

(5000, 30)

1. gsc_impressions
Knowable because impressions are historical observations already available at decision time.

2. gsc_clicks
Knowable because clicks have already occurred before refresh decisions are made.

3. gsc_avg_position
Knowable because ranking information is already observed.

4. ga4_pageviews
Knowable because pageview information already exists before intervention.

5. sessions_organic
Knowable because historical traffic-source information is available at the decision moment.

In [76]:
features = sample[
    [
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position",
        "ga4_pageviews",
        "sessions_organic"
    ]
]

features.head()

,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,sessions_organic
0,30,0,3.833333,0,0
1,5,0,71.600000,0,0
2,1,0,34.000000,0,0
3,6,0,23.333333,0,0
4,5,0,17.800000,0,0


A deliberately bad feature is created below to demonstrate leakage.

Because it is strongly tied to future performance, it can artificially inflate model quality and produce unrealistically optimistic results.

In [77]:
sample["leak_feature"] = (
    sample["gsc_clicks"]
    >
    sample["gsc_clicks"].median()
).astype(int)

sample[
    [
        "gsc_clicks",
        "leak_feature"
    ]
].head()

,gsc_clicks,leak_feature
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0


In [78]:
sample.drop(
    columns=["leak_feature"],
    inplace=True
)

## 4. Data limits

1. Client histories are unbalanced.

2. Availability of GSC and GA4 differs across clients.

3. Historical windows may overlap and create dependencies between observations.

4. The warehouse is observational.

Therefore, this dataset can support observed, measured, and directional insights, but it cannot prove that refreshing content causes ranking improvements.

The outputs should be interpreted as decision-support signals rather than causal evidence.

In [79]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(
    "Rows with GSC available:",
    sample[
        sample["gsc_data_available"] == True
    ].shape[0]
)

print(
    "Rows with GA4 available:",
    sample[
        sample["ga4_data_available"] == True
    ].shape[0]
)

Rows with GSC available: 5000
Rows with GA4 available: 0


✅ Every section above is filled with both reasoning and supporting code.

✅ The notebook runs top to bottom without errors.

✅ No client names, URLs, or private queries are present.

✅ Claims use careful wording:
observed, measured, directional, decision-support.

✅ Notebook committed under:

work/notebooks/w03_data_contract.ipynb